In [111]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio 
import seaborn as sns
import numpy as np 


pio.renderers.default = "notebook_connected"

In [112]:
df = pd.read_csv("data/database_without_names.csv",header=1)
df = df.drop(["Unnamed: 16","Unnamed: 17","Unnamed: 24"],axis=1)
df = df[~df["Domaine"].isna()]
df

,Domaine,Sous-domaine,Mot,N°ID,N° Signe,Définition,Explication,Date,Nom,Prénom,...,Format,Clarté,Total,Remarques,Téléchargé,Monté,Miniature,YouTube,Site Web,Lien YouTube
0,Informatique,Algorithmique,Débogage,1,1.0,non,non,15/12/2017,NaN,NaN,...,1.0,1.0,2.0,Refait par STIM x,oui,oui,oui,oui,oui,https://youtu.be/JfaMVjDwFa8
1,Informatique,Algorithmique,Ordonnancement,2,1.0,non,non,15/12/2017,NaN,NaN,...,1.0,1.0,2.0,Refait par STIM x,oui,oui,oui,oui,oui,https://youtu.be/f4dZizP-8_o
2,Informatique,Algorithmique,Thread,3,1.0,non,non,15/12/2017,NaN,NaN,...,1.0,1.0,2.0,Refait par STIM x,oui,oui,oui,oui,non,https://youtu.be/UA00dCHkF-Q
3,Chimie,Analytique,Spectromètre de masse,4,1.0,non,oui,15/12/2017,NaN,NaN,...,1.0,1.0,2.0,NaN,oui,oui,oui,oui,non,https://youtu.be/LTSTHkMN4GY
4,Physique,Classique,Aimant,5,1.0,non,non,15/12/2017,NaN,NaN,...,1.0,1.0,2.0,Refait par STIM x,oui,oui,oui,oui,oui,https://youtu.be/ASuaVlK8NIQ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1841,Environnement,Géologie,Faille transformante,1842,1.0,oui,oui,25/04/2026,NaN,NaN,...,1.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1842,Environnement,Géologie,Fissure,1843,1.0,oui,oui,25/04/2026,NaN,NaN,...,1.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1843,Environnement,Géologie,Sismogramme,1844,1.0,oui,oui,25/04/2026,NaN,NaN,...,1.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1844,Environnement,Géologie,Sismomètre,1845,1.0,oui,oui,25/04/2026,NaN,NaN,...,1.0,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [113]:
# Compte des domaines
dom = df.groupby(["Domaine","Sous-domaine"])["N°ID"].count().reset_index()
dom["N°ID"] = pd.to_numeric(dom["N°ID"])
# dom = dom[dom["N°ID"] > 10] 
dom["N°ID"]

0       2
1       6
2     171
3       2
4      10
5      19
6      12
7      22
8     190
9     134
10      5
11      5
12     20
13     13
14      5
15     15
16     54
17     18
18     23
19      2
20     24
21     26
22     23
23     27
24      3
25     16
26     19
27     20
28    100
29     34
30     45
31      1
32     26
33     53
34     13
35     22
36     34
37     93
38    146
39     32
40     25
41     24
42     54
43     23
44     64
45     73
46     34
47      1
48     16
49      4
50     10
51     22
52     11
Name: N°ID, dtype: int64

In [114]:
# n_colors = dom.shape[0]
n_colors = len(set(dom["Domaine"]))
color_scale = px.colors.sample_colorscale("plasma", [n/(n_colors -1) for n in range(n_colors)])
fig = px.treemap(dom,values="N°ID",path=["Domaine","Sous-domaine"],
                 color_discrete_sequence=color_scale, 
                 title="Répartition du signaire par domaines et sous-domaines",
                 width=1500, height=1000)
fig.update_traces(marker=dict(cornerradius=5),
                  root_color="black",
                  texttemplate='<b>%{label}</b><br>%{value}',
                  branchvalues="total",
                  textinfo='label+value',
                  textfont_size = 25)
fig.update_layout(margin = dict(t=60, l=25, r=25, b=25),
                  paper_bgcolor="black",
                  plot_bgcolor="black",
                  title=dict(text="Répartition du signaire par domaines et sous-domaines",
                             y=0.98,
                             x=0.5,
                             xanchor='center',
                             yanchor='top',
                             font=dict(size=45,
                                       color="white")))

#fig.data[0].textinfo = 'label+text+value'
fig.write_html("figures/treemap_domains.html")
# pio.get_chrome()
# fig.write_image("figures/treemap_domains.svg")
fig.show()

In [115]:
df.loc[df["Date"] == "29/02/2022", "Date"] = "28/02/2022"
df["Date"] = pd.to_datetime(df["Date"],dayfirst=True)
df.sort_values("Date")
df["Année"] = df["Date"].dt.year
df
dates = df.groupby(["Date"])["N°ID"].count().reset_index()
dates["cumsum"] = dates["N°ID"].cumsum()
dates

,Date,N°ID,cumsum
0,2017-12-15,44,44
1,2017-12-16,15,59
2,2018-05-13,81,140
3,2018-10-13,12,152
4,2018-10-14,57,209
5,2019-05-26,90,299
6,2020-03-26,2,301
7,2020-03-30,1,302
8,2020-04-05,1,303
9,2020-04-13,5,308


In [116]:
fig = px.line(dates, x="Date",y="cumsum", 
              title='Evolution du nombre de signes au cours du temps',
              width=1500, height=1000)

fig.update_layout(margin = dict(t=60, l=25, r=25, b=25),
                  paper_bgcolor="#112760",
                  plot_bgcolor="#112760",
                  xaxis_title=dict(font=dict(size=20,color="white")),
                  yaxis_title=dict(text="Somme cumulée",font=dict(size=20,color="white")),
                  xaxis=dict(tickfont=dict(size=16,color="white")),
                  yaxis=dict(tickfont=dict(size=16,color="white")),
                  title=dict(text="Evolution du nombre de signes au cours du temps",
                             y=0.98,
                             x=0.5,
                             xanchor='center',
                             yanchor='top',
                             font=dict(size=25,
                                       color="white")))

fig.update_traces(line_color='#FF7011', line_width=5)

fig.write_html("figures/signes_au_cours_du_temps.html")
# pio.get_chrome()
# fig.write_image("figures/signes_au_cours_du_temps.svg")
fig.show()

In [117]:
# Compte des domaines en fonction de l'année de création
dom_pers = df.groupby(["Domaine","Année"])["N°ID"].count().reset_index()
dom_pers = dom_pers.sort_values(["Domaine","Année"],ascending=True)

dom_pers["N°ID"] = pd.to_numeric(dom_pers["N°ID"])
n_colors = len(set(dom_pers["Domaine"]))
color_scale = px.colors.sample_colorscale("plasma", [n/(n_colors -1) for n in range(n_colors)])
fig = px.treemap(dom_pers,values="N°ID",path=["Domaine","Année"],
                 color_discrete_sequence=color_scale, 
                 title="Répartition du signaire par domaines et année de création",
                 width=1500, height=1000)
fig.update_traces(marker=dict(cornerradius=5),
                  root_color="black",
                  texttemplate='<b>%{label}</b><br>%{value}',
                  branchvalues="total",
                  textinfo='label+value',
                  textfont_size = 25)
fig.update_layout(margin = dict(t=60, l=25, r=25, b=25),
                  paper_bgcolor="black",
                  plot_bgcolor="black",
                  title=dict(text="Répartition du signaire par domaines et année de création",
                             y=0.98,
                             x=0.5,
                             xanchor='center',
                             yanchor='top',
                             font=dict(size=45,
                                       color="white")))

fig.data[0].textinfo = 'label+text+value'
fig.write_html("figures/treemap_domains_annee.html")
# pio.get_chrome()
# fig.write_image("figures/treemap_domains_annee.svg")
fig.show()

In [118]:
# Compte des domaines en fonction de l'année de création => inversé
dom_pers = dom_pers.sort_values(["Année","Domaine"],ascending=True)
n_colors = len(set(dom_pers["Année"]))
color_scale = px.colors.sample_colorscale("plasma", [n/(n_colors -1) for n in range(n_colors)])
fig = px.treemap(dom_pers,values="N°ID",path=["Année","Domaine"],
                 color_discrete_sequence=color_scale, 
                 title="Répartition du signaire par domaines et année de création",
                 width=1500, height=1000)
fig.update_traces(marker=dict(cornerradius=5),
                  root_color="black",
                  texttemplate='<b>%{label}</b><br>%{value}',
                  branchvalues="total",
                  textinfo='label+value',
                  textfont_size = 25)
fig.update_layout(margin = dict(t=60, l=25, r=25, b=25),
                  paper_bgcolor="black",
                  plot_bgcolor="black",
                  title=dict(text="Répartition du signaire par domaines et année de création",
                             y=0.98,
                             x=0.5,
                             xanchor='center',
                             yanchor='top',
                             font=dict(size=45,
                                       color="white")))

fig.data[0].textinfo = 'label+text+value'
fig.write_html("figures/treemap_annee_domains.html")
# pio.get_chrome()
# fig.write_image("figures/treemap_annee_domains.svg")
fig.show()